In [ ]:
!pip install -q --upgrade google-cloud-vision

In [ ]:
%env "# REMOVED: GCP service account key not needed for local execution"

In [ ]:
import pandas as pd

df = pd.read_csv('../data/2023-10-02-Stories-Person-Focus.csv')
df['image'] = df['image'].apply(lambda x: x.replace('/content/images/upload_s3/', ''))

In [ ]:
#@title Sign Image Links
from google.cloud import storage
import datetime

client = storage.Client.from_service_account_json("# REMOVED: GCP service account key not needed for local execution")

def signed_url(url, days=7):
  # Removing the 'gs://' prefix
  stripped_url = url.replace("gs://", "")

  # Splitting at the first '/'
  split_url = stripped_url.split("/", 1)

  bucket_name = "ig-politik-stories"
  blob_name = stripped_url

  bucket = client.bucket(bucket_name)
  blob = bucket.blob(blob_name)


  url = blob.generate_signed_url(
        version="v4",
        # This URL is valid for 15 minutes
        expiration=datetime.timedelta(days=days),
        # Allow GET requests using this URL.
        method="GET")
  return url

df['storage_url'] = df['image'].apply(signed_url)

In [ ]:
import pandas as pd
from google.cloud import vision
from tqdm.notebook import tqdm


def localize_objects_uri(df):
    """Localize objects in images on Google Cloud Storage using a dataframe.

    Args:
    df: A dataframe containing a column 'image' with paths to the files.

    Returns:
    DataFrame with columns: image, object_name, confidence, and vertices
    """
    client = vision.ImageAnnotatorClient()
    results = []

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        uri = "../data/" + row['image']
        image = vision.Image()
        image.source.image_uri = uri

        objects = client.object_localization(image=image).localized_object_annotations

        for object_ in objects:
            vertices = [{"x": vertex.x, "y": vertex.y} for vertex in object_.bounding_poly.normalized_vertices]
            result = {
                "image": row['image'],
                "object_name": object_.name,
                "confidence": object_.score,
                "vertices": vertices
            }
            results.append(result)

    return pd.DataFrame(results)

output_df = localize_objects_uri(df)
output_df.to_csv('../data/2023-10-02-Story-Google-Cloud-Vision-Object-Detection.csv')

In [ ]:
output_df

,image,object_name,confidence,vertices
0,armin_laschet_17-09-2021_00:00_image12.jpeg,Person,0.829303,"[{'x': 0.6684036254882812, 'y': 0.478167682886..."
1,armin_laschet_17-09-2021_00:00_image12.jpeg,Person,0.802489,"[{'x': 0.26122626662254333, 'y': 0.46860751509..."
2,armin_laschet_17-09-2021_00:00_image12.jpeg,Person,0.786831,"[{'x': 0.1409202367067337, 'y': 0.471591085195..."
3,armin_laschet_17-09-2021_00:00_image12.jpeg,Person,0.580025,"[{'x': 0.6197669506072998, 'y': 0.480672359466..."
4,armin_laschet_17-09-2021_00:00_image12.jpeg,Person,0.530636,"[{'x': 0.3476751744747162, 'y': 0.477527946233..."
5,armin_laschet_17-09-2021_00:00_image12.jpeg,Person,0.508690,"[{'x': 0.18321795761585236, 'y': 0.47055473923..."
6,armin_laschet_17-09-2021_00:00_image12.jpeg,Suit,0.502804,"[{'x': 0.6662693619728088, 'y': 0.487029045820..."
